In [7]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
from typing import Dict
import os
import sys

PROJECT_ROOT = os.path.abspath("../..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.embedder import SPARQLEmbedder

sys.path.append(os.path.abspath('..'))
from utils import (
    avg_prf_at_k,
    avg_mrr_at_k,
    avg_hits_at_k,
    avg_jaccard_at_k,
    avg_ndcg_at_k,
)

CHECKPOINT_PATH = "../../checkpoints/wikidata_model_semantic-20.pt"
BATCH_SIZE = 32
TOPKS = [3, 5, 10, 20]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
file_path = "../dataset/hard_triplets_tfidf_wikidata.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Successfully loaded {len(data)} query groups from the JSON file.")


Successfully loaded 186 query groups from the JSON file.


In [3]:
unique_queries = set()
eval_items     = []

for item in data:
    anchor    = item["anchor_query"]
    positives = item["positive_queries"]
    negatives = item["negative_queries"]

    pool = positives + negatives

    unique_queries.add(anchor)
    unique_queries.update(positives)
    unique_queries.update(negatives)

    eval_items.append((anchor, pool, set(positives)))

queries_list = list(unique_queries)
print(f"Loaded {len(eval_items)} evaluation items.")
print(f"Unique queries in vocabulary: {len(queries_list)}")


Loaded 186 evaluation items.
Unique queries in vocabulary: 2229


In [4]:
# Binary relevance: 1.0 if candidate is in positive_queries, else 0.0.
# No SPARQL endpoint, no cache, and no safe_jaccard_for_queries needed.
# Membership in positive_queries already encodes Jaccard >= threshold (decided at build time).

relevance_scores_per_item = []

for _, pool, positives in eval_items:
    scores = {cand: (1.0 if cand in positives else 0.0) for cand in pool}
    relevance_scores_per_item.append(scores)

print(f"Binary relevance scores built for {len(relevance_scores_per_item)} items. No SPARQL calls needed.")


Binary relevance scores built for 186 items. No SPARQL calls needed.


In [5]:
def build_hybrid_embedding_map(
    model, queries: list, batch_size: int = 32, normalize_embeddings: bool = True
) -> Dict[str, np.ndarray]:
    """Encodes all queries exactly once with the custom SPARQLCE encoder.
    Applies manual L2 normalisation (required for accurate dot-product ranking).
    """
    embedding_map: Dict[str, np.ndarray] = {}
    for i in range(0, len(queries), batch_size):
        batch = queries[i : i + batch_size]
        embs  = model.encode(batch)
        if not isinstance(embs, np.ndarray):
            embs = np.array(embs)
        if normalize_embeddings:
            norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12
            embs  = embs / norms
        for q, e in zip(batch, embs):
            embedding_map[q] = e
    return embedding_map


def build_sorted_pools(emb_map: Dict[str, np.ndarray], items: list) -> list:
    """Ranks each candidate pool by dot-product similarity to the anchor."""
    sorted_pools = []
    for anchor_query, pool, _ in items:
        pool_embs  = np.vstack([emb_map[q] for q in pool])
        anchor_emb = emb_map[anchor_query]
        scores     = pool_embs @ anchor_emb
        order      = np.argsort(-scores)
        sorted_pools.append([pool[i] for i in order])
    return sorted_pools


In [8]:
print(f"Loading SPARQLCE from: {CHECKPOINT_PATH}")
sparqlce_model = SPARQLEmbedder(CHECKPOINT_PATH)

print("Encoding queries...")
sparqlce_emb_map = build_hybrid_embedding_map(
    model=sparqlce_model,
    queries=list(unique_queries),
    batch_size=BATCH_SIZE,
)

print("Ranking pools...")
sparqlce_sorted_pools = build_sorted_pools(sparqlce_emb_map, eval_items)

print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS: SPARQLCE")
print("=" * 60)

print("\n[Binary relevance: Annotated Positives]")
print(f"{'':5} {'P':>7} {'R':>7} {'F1':>7} {'MRR':>7} {'Hits':>7} {'avgJacc':>9} {'nDCG':>7}")
print("-" * 62)

for k in TOPKS:
    p, r, f1 = avg_prf_at_k(sparqlce_sorted_pools, eval_items, k)
    mrr = avg_mrr_at_k(sparqlce_sorted_pools, eval_items, k)
    hits = avg_hits_at_k(sparqlce_sorted_pools, eval_items, k)
    avg_jacc = avg_jaccard_at_k(sparqlce_sorted_pools, k, relevance_scores_per_item)
    ndcg = avg_ndcg_at_k(sparqlce_sorted_pools, k, relevance_scores_per_item)

    print(f"@{k:<4} {p:>7.4f} {r:>7.4f} {f1:>7.4f} {mrr:>7.4f} {hits:>7.4f} {avg_jacc:>9.4f} {ndcg:>7.4f}")

Loading SPARQLCE from: ../../checkpoints/wikidata_model_semantic-20.pt


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 37210.40it/s]


Loading model from ../../checkpoints/wikidata_model_semantic-20.pt...
Encoding queries...
Ranking pools...

FINAL EVALUATION RESULTS: SPARQLCE

[Binary relevance: Annotated Positives]
            P       R      F1     MRR    Hits   avgJacc    nDCG
--------------------------------------------------------------
@3     0.8172  0.3342  0.4265  0.8262  0.9194    0.8172  0.8015
@5     0.8075  0.5236  0.5640  0.8391  0.9731    0.8086  0.8492
@10    0.7102  0.7868  0.6646  0.8434  1.0000    0.7134  0.8833
@20    0.5046  0.9496  0.5889  0.8434  1.0000    0.5099  0.9057
